In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

# -----------------------------
# DISPLAY SETTINGS
# -----------------------------
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# -----------------------------
# STEP 0: LOAD DATA
# -----------------------------
df = pd.read_csv('../data/zomato.csv', encoding='latin-1')
df_country = pd.read_excel('../data/Country-Code.xlsx')

# -----------------------------
# STEP 1: RENAME COLUMNS
# -----------------------------
df = df.rename(columns={
    'Restaurant ID': 'restaurant_id',
    'Restaurant Name': 'restaurant_name',
    'Country Code': 'country_code',
    'City': 'city',
    'Locality': 'location',
    'Cuisines': 'cuisine',
    'Average Cost for two': 'price',
    'Aggregate rating': 'rating',
    'Votes': 'votes'
})

df_country = df_country.rename(columns={
    'Country Code': 'country_code',
    'Country': 'country'
})

# -----------------------------
# STEP 2: MERGE COUNTRY DATA
# -----------------------------
df = df.merge(df_country, on='country_code', how='left')

# Keep only India
df = df[df['country'] == 'India']

# -----------------------------
# STEP 3: CLEAN DATA
# -----------------------------
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['cuisine', 'votes', 'rating', 'price'])

# -----------------------------
# STEP 4: COUNT CUISINES
# -----------------------------
df['num_cuisines'] = df['cuisine'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) else 1
)

# -----------------------------
# STEP 5: ADJUST DEMAND
# -----------------------------
df['adjusted_demand'] = df['votes'] / df['num_cuisines']

# -----------------------------
# STEP 6: EXPLODE CUISINES
# -----------------------------
df['cuisine'] = df['cuisine'].str.split(',')
df = df.explode('cuisine')
df['cuisine'] = df['cuisine'].str.strip()

# -----------------------------
# STEP 7: AGGREGATE
# -----------------------------
combo = df.groupby(['location', 'cuisine']).agg({
    'restaurant_id': 'nunique',
    'adjusted_demand': 'sum',
    'rating': 'mean',
    'price': 'mean'
}).rename(columns={
    'restaurant_id': 'competition',
    'adjusted_demand': 'demand'
}).reset_index()

# -----------------------------
# STEP 8: OPPORTUNITY SCORE
# -----------------------------
combo['opportunity_score'] = (
    np.log1p(combo['demand']) * (combo['rating'] / 5)
) / (combo['competition'] + 5)

# -----------------------------
# STEP 9: NORMALIZE DEMAND
# -----------------------------
scaler = MinMaxScaler()
combo['normalized_demand'] = scaler.fit_transform(combo[['demand']])

# -----------------------------
# STEP 10: TOP CUISINES PER LOCATION
# -----------------------------
best_per_location = combo.sort_values(
    ['location', 'opportunity_score'], 
    ascending=[True, False]
).groupby('location').head(3)

# -----------------------------
# STEP 11: LOCATION SCORES
# -----------------------------
location_scores = combo.groupby('location')['opportunity_score']\
                       .mean().reset_index()

location_scores = location_scores.sort_values(
    'opportunity_score', ascending=False
)

# -----------------------------
# STEP 12: CLEAN DISPLAY TABLE
# -----------------------------
df_show = best_per_location.copy()

df_show = df_show[
    ['location', 'cuisine', 'competition', 'demand', 'rating', 'price', 'opportunity_score']
]

df_show = df_show.round(2)

# Add ranking per location
df_show['rank'] = df_show.groupby('location')['opportunity_score']\
                        .rank(ascending=False, method='first')

df_show = df_show.sort_values(['location', 'rank'])

# -----------------------------
# OUTPUT (STYLED)
# -----------------------------
print("\n🔥 Top cuisines per location:")
display(
    df_show.head(20).style
    .background_gradient(subset=['opportunity_score'], cmap='Greens')
    .set_properties(**{'text-align': 'left'})
)

print("\n🏆 Best locations overall:")
display(location_scores.head(10).round(3))

# -----------------------------
# SAVE OUTPUTS
# -----------------------------
combo.to_csv('../data/combo_output.csv', index=False)
best_per_location.to_csv('../data/best_per_location.csv', index=False)
location_scores.to_csv('../data/location_scores.csv', index=False)


🔥 Top cuisines per location:


,location,cuisine,competition,demand,rating,price,opportunity_score,rank
2,"ILD Trade Centre Mall, Sohna Road",Mughlai,1,40.000000,2.700000,800.000000,0.330000,1.000000
3,"ILD Trade Centre Mall, Sohna Road",North Indian,1,40.000000,2.700000,800.000000,0.330000,2.000000
0,"ILD Trade Centre Mall, Sohna Road",Beverages,1,8.000000,3.400000,350.000000,0.250000,3.000000
4,"12th Square Building, Banjara Hills",Chinese,1,1124.670000,4.300000,1500.000000,1.010000,1.000000
5,"12th Square Building, Banjara Hills",Mughlai,1,1124.670000,4.300000,1500.000000,1.010000,2.000000
6,"12th Square Building, Banjara Hills",North Indian,1,1124.670000,4.300000,1500.000000,1.010000,3.000000
7,"A Hotel, Gurdev Nagar",Chinese,1,31.000000,3.600000,800.000000,0.420000,1.000000
8,"A Hotel, Gurdev Nagar",Fast Food,1,31.000000,3.600000,800.000000,0.420000,2.000000
9,"A Hotel, Gurdev Nagar",North Indian,1,31.000000,3.600000,800.000000,0.420000,3.000000
10,"ARSS Mall, Paschim Vihar",Chinese,1,23.400000,3.100000,500.000000,0.330000,1.000000



🏆 Best locations overall:


,location,opportunity_score
155,Deccan Gymkhana,1.142
48,Azad Nagar,1.139
331,Kilpauk,1.122
526,"Riverside Mall, Gomti Nagar",1.093
497,"R Deccan Mall, JM Road",1.089
103,"Chokhi Dhani Village Resort, Tonk Road",1.046
494,"Quest Mall, Ballygunge",1.029
21,Aminabad,1.024
396,Marathahalli,1.016
435,Nungambakkam,1.012


In [4]:
import pandas as pd
df = pd.read_csv("../data/location_scores.csv")
print(df.columns)

Index(['location', 'opportunity_score'], dtype='object')
